## Challenge 2: House Price - Train after FE

### Import thư viện và đọc dữ liệu

In [1]:
import os
import sys
import random
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
from IPython import display
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Machine Learning
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Regression Models
from sklearn.ensemble import (
    RandomForestRegressor,
    ExtraTreesRegressor,
    GradientBoostingRegressor
)
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.tree import DecisionTreeRegressor

# Boosting Libraries
from xgboost import XGBRegressor
from lightgbm import LGBMRegressor

# Ignore warnings
import warnings
warnings.filterwarnings("ignore")

Tham số thực nghiệm

In [2]:
params = {}

# Thư mục thí nghiệm
params["exps_dir"]  = "../exps"
params["exp_name"]  = "challenge2_houseprice_standard"
params["save_dir"]  = f'{params["exps_dir"]}/result1_{params["exp_name"]}'

# Đường dẫn dữ liệu đã preprocess (before FE)
params["data_path"] = f'{params["exps_dir"]}/data/train_pre_fe.xlsx'
params["test_path"] = f'{params["exps_dir"]}/data/test_pre_fe.xlsx'

params["k_fold"] = 10
params["random_state"] = 42

os.makedirs(params["save_dir"], exist_ok=True)

random.seed(params["random_state"])
np.random.seed(params["random_state"])
os.environ["PYTHONHASHSEED"] = str(params["random_state"])

print("Save dir:", params["save_dir"])
print("Train path:", params["data_path"])
print("Test path :", params["test_path"])

Save dir: ../exps/result1_challenge2_houseprice_standard
Train path: ../exps/data/train_pre_fe.xlsx
Test path : ../exps/data/test_pre_fe.xlsx


In [3]:
# Đọc dữ liệu sau Feature Engineering
train = pd.read_excel(params["data_path"])
test  = pd.read_excel(params["test_path"])

print("Đọc dữ liệu thành công:", train.shape, test.shape)
train.head()

Đọc dữ liệu thành công: (1460, 226) (1459, 225)


,MSSubClass,LotFrontage,LotArea,OverallQual,OverallCond,YearBuilt,YearRemodAdd,MasVnrArea,BsmtFinSF1,BsmtFinSF2,...,SaleType_New,SaleType_Oth,SaleType_WD,SaleCondition_Abnorml,SaleCondition_AdjLand,SaleCondition_Alloca,SaleCondition_Family,SaleCondition_Normal,SaleCondition_Partial,SalePrice
0,0.424462,-0.083837,-0.133270,0.651479,-0.517200,1.050994,0.878668,1.203619,0.779431,-0.355342,...,0,0,1,0,0,0,0,1,0,208500
1,-1.125202,0.548841,0.113413,-0.071836,2.179628,0.156734,-0.429577,-0.806841,0.888257,-0.355342,...,0,0,1,0,0,0,0,1,0,181500
2,0.424462,0.053489,0.420049,0.651479,-0.517200,0.984752,0.830215,1.131524,0.654803,-0.355342,...,0,0,1,0,0,0,0,1,0,223500
3,0.645073,-0.327217,0.103317,0.651479,-0.517200,-1.863632,-0.720298,-0.806841,0.384539,-0.355342,...,0,0,1,1,0,0,0,0,0,140000
4,0.424462,0.697753,0.878431,1.374795,-0.517200,0.951632,0.733308,1.423411,0.754400,-0.355342,...,0,0,1,0,0,0,0,1,0,250000


In [4]:
train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Columns: 226 entries, MSSubClass to SalePrice
dtypes: float64(58), int64(168)
memory usage: 2.5 MB


### Kiểm tra dữ liệu trước khi train

In [5]:
# Kiểm tra missing values trong train
missing_train = train.isnull().sum()
missing_train = missing_train[missing_train > 0].sort_values(ascending=False)  # Chỉ hiển thị các cột có missing values
print(f"Missing values in train data:\n{missing_train}")

Missing values in train data:
Series([], dtype: int64)


### Tách biến mục tiêu

In [6]:
# Tách X, y
y = train["SalePrice"]
X = train.drop(columns=["SalePrice"])


print("X shape:", X.shape)
print("y shape:", y.shape)

X_train, X_valid, y_train, y_valid = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
)

print("Train X:", X_train.shape)
print("Valid X:", X_valid.shape)
print("Train y:", y_train.shape)
print("Valid y:", y_valid.shape)

X shape: (1460, 225)
y shape: (1460,)
Train X: (1168, 225)
Valid X: (292, 225)
Train y: (1168,)
Valid y: (292,)


In [7]:
kfold = KFold(
    n_splits=params["k_fold"],
    shuffle=True,
    random_state=params["random_state"]
)

print(f"Total rows in X_train: {len(X_train)}\n")

for fold, (train_idx, valid_idx) in enumerate(kfold.split(X_train, y_train)):
    print(f"---- Fold {fold} ----")
    print(f"Train size: {len(train_idx)}")
    print(f"Valid size: {len(valid_idx)}")
    print(f"Train idx sample: {train_idx[:10]}")
    print(f"Valid idx sample: {valid_idx[:10]}")
    print()

Total rows in X_train: 1168

---- Fold 0 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [ 23  44  49  51  54  58  70  86 101 107]

---- Fold 1 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [10 31 43 56 59 63 76 83 88 96]

---- Fold 2 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1  4  5  7  8  9 10 11 13]
Valid idx sample: [ 2  3  6 12 25 27 30 39 47 55]

---- Fold 3 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1  2  3  4  6  7  8  9 10]
Valid idx sample: [ 5 29 33 60 65 71 77 82 84 92]

---- Fold 4 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 1  2  3  4  5  6  8 10 11 12]
Valid idx sample: [  0   7   9  62  69  79  81  90  97 104]

---- Fold 5 ----
Train size: 1051
Valid size: 117
Train idx sample: [0 1 2 3 4 5 6 7 8 9]
Valid idx sample: [11 15 18 24 28 41 42 61 73 74]

---- Fold 6 ----
Train size: 1051
Valid size: 117
Train idx sample: [ 0  1 

In [8]:
models = [
    ("Extra Trees", ExtraTreesRegressor(random_state=params["random_state"])),
    ("Random Forest", RandomForestRegressor(random_state=params["random_state"])),
    ("LightGBM", LGBMRegressor(
        random_state=params["random_state"],
        n_estimators=2000,
        learning_rate=0.03,
        subsample=0.8,
        colsample_bytree=0.8,
        verbose=-1
    )),
    ("Gradient Boosting", GradientBoostingRegressor(random_state=params["random_state"])),
    ("XGBoost", XGBRegressor(
        random_state=params["random_state"],
        n_estimators=3000,
        learning_rate=0.03,
        max_depth=3,
        subsample=0.8,
        colsample_bytree=0.8,
        objective="reg:squarederror"
    ))
]

### Train & đánh giá nhiều mô hình

In [9]:
results = []
baseline_results = {}

y_log = np.log1p(y)

for name, model in models:
    print(f"===== Model: {name} =====")

    baseline_results[name] = {"mae": [], "rmse": [], "r2": []}

    kf = KFold(
        n_splits=params["k_fold"],
        shuffle=True,
        random_state=params["random_state"]
    )

    for fold, (train_idx, valid_idx) in enumerate(kf.split(X, y_log)):
        X_tr, y_tr = X.iloc[train_idx], y_log.iloc[train_idx]
        X_val, y_val = X.iloc[valid_idx], y_log.iloc[valid_idx]

        # Train
        model.fit(X_tr, y_tr)

        # Predict (log-space)
        y_pred = model.predict(X_val)

        # Convert back for metrics
        y_pred_real = np.expm1(y_pred)
        y_val_real  = np.expm1(y_val)

        mae = mean_absolute_error(y_val_real, y_pred_real)
        rmse = np.sqrt(mean_squared_error(y_val_real, y_pred_real))
        r2 = r2_score(y_val_real, y_pred_real)

        baseline_results[name]["mae"].append(mae)
        baseline_results[name]["rmse"].append(rmse)
        baseline_results[name]["r2"].append(r2)

        print(f" Fold {fold}: RMSE = {rmse:.4f}")

    print(f"--> Mean RMSE: {np.mean(baseline_results[name]['rmse']):.4f} ± {np.std(baseline_results[name]['rmse']):.4f}\n")

    results.append([
        name,
        np.mean(baseline_results[name]["mae"]),
        np.mean(baseline_results[name]["rmse"]),
        np.mean(baseline_results[name]["r2"])
    ])

===== Model: Extra Trees =====
 Fold 0: RMSE = 27306.5601
 Fold 1: RMSE = 23266.9091
 Fold 2: RMSE = 20086.6997
 Fold 3: RMSE = 28094.8244
 Fold 4: RMSE = 42477.6859
 Fold 5: RMSE = 27260.6283
 Fold 6: RMSE = 27846.4700
 Fold 7: RMSE = 26124.4278
 Fold 8: RMSE = 22929.0499
 Fold 9: RMSE = 21765.1220
--> Mean RMSE: 26715.8377 ± 5893.2269

===== Model: Random Forest =====
 Fold 0: RMSE = 32526.4811
 Fold 1: RMSE = 24195.0672
 Fold 2: RMSE = 19924.9486
 Fold 3: RMSE = 33059.9105
 Fold 4: RMSE = 39867.0740
 Fold 5: RMSE = 32302.9157
 Fold 6: RMSE = 30730.7073
 Fold 7: RMSE = 26690.8198
 Fold 8: RMSE = 25241.0673
 Fold 9: RMSE = 21756.0248
--> Mean RMSE: 28629.5016 ± 5804.5254

===== Model: LightGBM =====
 Fold 0: RMSE = 32027.5356
 Fold 1: RMSE = 25124.4521
 Fold 2: RMSE = 21868.8327
 Fold 3: RMSE = 33935.8447
 Fold 4: RMSE = 30057.7228
 Fold 5: RMSE = 29273.4284
 Fold 6: RMSE = 30227.5678
 Fold 7: RMSE = 23891.9342
 Fold 8: RMSE = 22463.3155
 Fold 9: RMSE = 16953.0599
--> Mean RMSE: 26582

In [10]:
df_results = pd.DataFrame(
    results,
    columns=["Model", "Mean_MAE", "Mean_RMSE", "Mean_R2"]
)

# Sắp xếp theo RMSE thấp nhất (mục tiêu RMSE càng thấp càng tốt)
df_results = df_results.sort_values(by="Mean_RMSE", ascending=True)

display.display(df_results)

,Model,Mean_MAE,Mean_RMSE,Mean_R2
4,XGBoost,14137.172167,24397.438285,0.897944
3,Gradient Boosting,15398.974130,25842.169178,0.886528
2,LightGBM,15684.872858,26582.369370,0.884693
0,Extra Trees,16051.055763,26715.837709,0.875236
1,Random Forest,16964.547582,28629.501620,0.861488
